# IPL SQL Analysis — Final Assignment

**Notebook:** `01b_sql_practice.ipynb`

This notebook contains the SQL queries required by the IPL SQL Analysis assignment.  
For each command, the query is followed by a short observation explaining what the SQL command demonstrates.

**Workflow:** COMMENT → SQL QUERY → RUN → RESULT → OBSERVATION


In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/raw/ipl.db")

def q(sql):
    return pd.read_sql(sql, con)

# Test the database connection
q("SELECT COUNT(*) AS total_matches FROM matches;")


## Part A — Basic Exploration

In [ ]:
# Command 1 — SELECT and FROM
q("""
SELECT *
FROM teams;
""")

# My observation:
# SELECT * displays all columns and rows from the teams table.


In [ ]:
# Command 2 — LIMIT
q("""
SELECT *
FROM deliveries
LIMIT 10;
""")

# My observation:
# LIMIT restricts the number of rows returned by the query.


In [ ]:
# Command 3 — Selecting Specific Columns
q("""
SELECT match_id, season, match_winner, result
FROM matches;
""")

# My observation:
# SELECT can be used to return only the columns needed for analysis.


In [ ]:
# Command 4 — DISTINCT
q("""
SELECT DISTINCT season
FROM matches
ORDER BY season;
""")

# My observation:
# DISTINCT removes duplicate values and shows each season only once.


In [ ]:
# Command 5 — DISTINCT cities
q("""
SELECT DISTINCT city
FROM matches
ORDER BY city;
""")

# My observation:
# DISTINCT shows the unique cities recorded in the matches table.


## Part B — Filtering and Sorting

In [ ]:
# Command 6 — WHERE
q("""
SELECT *
FROM matches
WHERE season = 2019;
""")

# My observation:
# WHERE filters the table so that only matches from the 2019 season are returned.


In [ ]:
# Command 7 — WHERE with a numeric condition
q("""
SELECT *
FROM matches
WHERE win_by_runs > 100;
""")

# My observation:
# The greater-than condition returns matches where the winning margin in runs exceeded 100.


In [ ]:
# Command 8 — Multiple WHERE conditions
q("""
SELECT *
FROM matches
WHERE season = 2016
  AND win_by_wickets > 5;
""")

# My observation:
# AND requires both conditions to be true, so only qualifying 2016 matches are returned.


In [ ]:
# Command 9 — ORDER BY
q("""
SELECT match_id, season, team1, team2, match_winner, win_by_runs
FROM matches
WHERE win_by_runs IS NOT NULL
ORDER BY win_by_runs DESC
LIMIT 10;
""")

# My observation:
# ORDER BY DESC sorts the winning margins from largest to smallest.


In [ ]:
# Command 10 — ORDER BY with NULL filtering
q("""
SELECT match_id, season, team1, team2, match_winner, win_by_wickets
FROM matches
WHERE win_by_wickets IS NOT NULL
ORDER BY win_by_wickets ASC
LIMIT 10;
""")

# My observation:
# IS NOT NULL removes missing wicket margins, and ASC displays the smallest margins first.


## Part C — Aggregation

In [ ]:
# Command 11 — COUNT and AS
q("""
SELECT COUNT(*) AS total_matches
FROM matches;
""")

# My observation:
# COUNT(*) counts the number of rows in the matches table, while AS gives the result a clear column name.


In [ ]:
# Command 12 — COUNT(*)
q("""
SELECT COUNT(*) AS total_deliveries
FROM deliveries;
""")

# My observation:
# COUNT(*) counts the total number of delivery records in the deliveries table.


In [ ]:
# Command 13 — COUNT(DISTINCT)
q("""
SELECT COUNT(DISTINCT venue) AS different_venues
FROM matches;
""")

# My observation:
# COUNT(DISTINCT venue) counts the different venue values recorded in matches.


In [ ]:
# Command 14 — GROUP BY and ORDER BY
q("""
SELECT venue, COUNT(*) AS matches_hosted
FROM matches
GROUP BY venue
ORDER BY matches_hosted DESC;
""")

# My observation:
# GROUP BY creates one group for each venue, and COUNT(*) counts matches in each group.


In [ ]:
# Command 15 — GROUP BY winners
q("""
SELECT match_winner, COUNT(*) AS number_of_wins
FROM matches
WHERE match_winner IS NOT NULL
GROUP BY match_winner
ORDER BY number_of_wins DESC;
""")

# My observation:
# GROUP BY creates one group for each match winner and COUNT(*) gives the number of wins.


In [ ]:
# Command 16 — HAVING
q("""
SELECT venue, COUNT(*) AS matches_hosted
FROM matches
GROUP BY venue
HAVING COUNT(*) >= 20
ORDER BY matches_hosted DESC;
""")

# My observation:
# HAVING filters grouped results and keeps only venues that hosted at least 20 matches.


In [ ]:
# Additional aggregation — Bowlers with more than 2,000 deliveries
q("""
SELECT bowler, COUNT(*) AS deliveries_bowled
FROM deliveries
GROUP BY bowler
HAVING COUNT(*) > 2000
ORDER BY deliveries_bowled DESC;
""")

# My observation:
# GROUP BY counts deliveries for each bowler, and HAVING keeps bowlers with more than 2,000 deliveries.


## Part D — NULL and Data Quality

In [ ]:
# Command — IS NULL: matches where city is NULL
q("""
SELECT COUNT(*) AS missing_city_matches
FROM matches
WHERE city IS NULL;
""")

# My observation:
# IS NULL correctly identifies rows where the city value is missing.


In [ ]:
# Command — IS NULL: matches where match_number is NULL
q("""
SELECT COUNT(*) AS missing_match_number
FROM matches
WHERE match_number IS NULL;
""")

# My observation:
# IS NULL identifies matches where match_number has no value.


In [ ]:
# Command — LOWER, TRIM and LENGTH
q("""
SELECT
    field_pos,
    TRIM(field_pos) AS trimmed_field_pos,
    LOWER(TRIM(field_pos)) AS normalized_field_pos,
    LENGTH(TRIM(field_pos)) AS field_pos_length
FROM deliveries
WHERE field_pos IS NOT NULL
LIMIT 50;
""")

# My observation:
# TRIM removes surrounding spaces, LOWER normalizes text to lowercase, and LENGTH shows the number of characters.


### Why `= NULL` is not the correct test

SQL uses three-valued logic for NULL. `NULL` represents a missing or unknown value, so a comparison such as `column = NULL` does not evaluate to TRUE. The correct tests are `IS NULL` and `IS NOT NULL`.


## Part E — JOIN and LEFT JOIN

In [ ]:
# Command — JOIN matches and venues
q("""
SELECT
    m.match_id,
    v.venue,
    m.city
FROM matches AS m
JOIN venues AS v
    ON m.venue = v.venue;
""")

# My observation:
# JOIN combines matching records from matches and venues using the venue field.


In [ ]:
# Command — JOIN deliveries and players
q("""
SELECT
    d.batter,
    p.bat_style
FROM deliveries AS d
JOIN players AS p
    ON d.batter = p.player_name
LIMIT 100;
""")

# My observation:
# The JOIN adds player information to delivery records by matching the batter name with the player table.


In [ ]:
# Command — LEFT JOIN + IS NULL: batters missing from players
q("""
SELECT DISTINCT
    d.batter
FROM deliveries AS d
LEFT JOIN players AS p
    ON d.batter = p.player_name
WHERE p.player_name IS NULL
ORDER BY d.batter;
""")

# My observation:
# LEFT JOIN keeps all delivery batters, and IS NULL identifies batters that have no matching player record.


In [ ]:
# Command — LEFT JOIN + IS NULL: match venues missing from venues
q("""
SELECT DISTINCT
    m.venue
FROM matches AS m
LEFT JOIN venues AS v
    ON m.venue = v.venue
WHERE v.venue IS NULL
ORDER BY m.venue;
""")

# My observation:
# LEFT JOIN keeps all match venues, and IS NULL identifies venues without a matching record in the venues table.


## Part F — CASE WHEN and EXCEPT

In [ ]:
# Command — CASE WHEN: create city_missing
q("""
SELECT
    match_id,
    city,
    CASE
        WHEN city IS NULL THEN 1
        ELSE 0
    END AS city_missing
FROM matches;
""")

# My observation:
# CASE WHEN creates a flag with 1 for missing cities and 0 when a city is present.


In [ ]:
# Command — total matches and missing-city matches
q("""
SELECT
    COUNT(*) AS total_matches,
    SUM(
        CASE
            WHEN city IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_city_matches
FROM matches;
""")

# My observation:
# COUNT(*) gives the total matches, while the CASE expression counts matches where city is missing.


In [ ]:
# Command — EXCEPT: venues never appearing in matches
q("""
SELECT venue
FROM venues

EXCEPT

SELECT venue
FROM matches;
""")

# My observation:
# EXCEPT returns venues that exist in the venues table but do not appear in the matches table.


In [ ]:
# Command — Reverse EXCEPT
q("""
SELECT venue
FROM matches

EXCEPT

SELECT venue
FROM venues;
""")

# My observation:
# Reversing EXCEPT changes the question: it finds venues appearing in matches that are absent from the venues table.


## Final Checklist

- Notebook filename: `01b_sql_practice.ipynb`
- Database connection works
- Required SQL commands are included
- Every query has a command comment
- Every query is executed
- Results are displayed by Jupyter
- Every query has a short observation
- Restart the kernel and run all cells before submission
- Check that there are no unresolved errors
